# 06-7. 타임아웃·오류·재시도 예제

## Goal

- 재시도 가능한 오류와 즉시 중단할 오류를 구분합니다.
- 대기 계획을 실제 수면 없이 검증합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

스크립트된 함수로 실패 순서를 고정해 같은 결과를 재현합니다.


## Steps

### 제한된 재시도 정책

시도 횟수와 대기 시간을 기록하며 마지막 오류를 숨기지 않습니다.


In [1]:
class TemporaryFailure(OSError):
    pass


def run_with_retry(operation, attempts=3, base_delay=0.25):
    history = []
    for number in range(1, attempts + 1):
        try:
            return operation(), history
        except TemporaryFailure as error:
            history.append({"attempt": number, "error": str(error)})
            if number == attempts:
                raise
            history[-1]["next_delay"] = base_delay * (2 ** (number - 1))


outcomes = iter([TemporaryFailure("일시 오류 1"), TemporaryFailure("일시 오류 2"), "성공"])


def scripted_operation():
    outcome = next(outcomes)
    if isinstance(outcome, Exception):
        raise outcome
    return outcome


value, retry_history = run_with_retry(scripted_operation)
print(value)
print(retry_history)


성공
[{'attempt': 1, 'error': '일시 오류 1', 'next_delay': 0.25}, {'attempt': 2, 'error': '일시 오류 2', 'next_delay': 0.5}]


## Checks

성공 시도 수와 지수형 대기 계획을 확인합니다.


In [2]:
assert value == "성공"
assert [item["attempt"] for item in retry_history] == [1, 2]
assert [item["next_delay"] for item in retry_history] == [0.25, 0.5]
print("재시도 정책 검사 통과")


재시도 정책 검사 통과


## Next Steps

실제 네트워크 코드에서는 전체 시간 한도, 멱등성, 서버의 재시도 지시를 함께 고려합니다.
